In [3]:
import numpy as np, pandas as pd
from scipy.optimize import lsq_linear

filename = "../estimation_timing/direct_enumeration/estimation_timing.pkl"
df = pd.read_pickle(filename)
df = df[df["seconds_total"].notna() & np.isfinite(df["seconds_total"]) & (df["seconds_total"] > 0)]
if "completed" in df: df = df[df["completed"] == 1.0]

avg = df.groupby("n")["seconds_total"].mean()
n, y = avg.index.to_numpy(float), avg.to_numpy(float)

for name, x in [("A * 2^n + B", 2.0 ** n), ("A * n^2 * 2^n + B", n**2 * 2.0 ** n)]:
    X = np.column_stack([x / y, 1.0 / y])
    A, B = lsq_linear(X, np.ones(len(y)), bounds=(0, np.inf)).x
    relative_RMSE = np.sqrt(np.mean(((y - (A * x + B)) / y) ** 2))
    print(f"{name}: A = {A:.6e}, B = {B:.6e}, relative_RMSE = {relative_RMSE:.6e}")

A * 2^n + B: A = 1.504584e-07, B = 1.348755e-23, relative_RMSE = 4.536564e-01
A * n^2 * 2^n + B: A = 6.323731e-10, B = 5.600878e-06, relative_RMSE = 2.559732e-01


We fit the direct-enumeration timing using two ansatzes, $A2^n + B$ and $An^2 2^n + B$, with $A,B \ge 0$. The best fit is selected by minimizing the relative RMSE, defined as 
$$\operatorname{rel\_RMSE} = \sqrt{\frac{1}{N}\sum_{i=1}^{N}\left(\frac{y_i-\hat{y}_i}{y_i}\right)^2},$$ 
where $y_i$ is the observed timing and $\hat y_i$ is the fitted timing.

The fit results are:

| Model          |                        $A$ |                        $B$ |             relative RMSE |
| -------------- | -------------------------: | -------------------------: | ------------------------: |
| $A2^n + B$     |  $1.504584 \times 10^{-7}$ | $1.348755 \times 10^{-23}$ | $4.536564 \times 10^{-1}$ |
| $An^2 2^n + B$ | $6.323731 \times 10^{-10}$ |  $5.600878 \times 10^{-6}$ | $2.559732 \times 10^{-1}$ |

As expected, $An^2 2^n + B$ gives the better fit. This is consistent with the implementation, where the enumeration visits all $2^n$ spin configurations and, for each configuration, recomputes the full dense Ising energy, whose cost scales as $O(n^2)$.


We consider a finite Markov chain $(\Omega,P_\beta,q_0)$ on the spin configuration space $\Omega={\pm1}^n$, where $q_0$ is an efficiently preparable initial distribution and $P_\beta$ is a transition kernel targeting the Gibbs state $\pi_\beta(x)\propto e^{-\beta H(x)}$. Given the current configuration $x_t$, a trial move $y$ is sampled from a proposal kernel $Q(y\mid x_t)$ and accepted with probability $A_\beta(y,x_t)$. For symmetric proposals, $Q(y\mid x)=Q(x\mid y)$, the Metropolis rule is $A_\beta(y,x)=\min{1,e^{-\beta(H(y)-H(x))}}$. Thus $x_{t+1}=y$ if the move is accepted, while $x_{t+1}=x_t$ otherwise. We restrict to ergodic and reversible chains; see Appendix~\ref{app:mcmc}. Reversibility imposes detailed balance, $\pi_\beta(x)p_\beta(y\mid x)=\pi_\beta(y)p_\beta(x\mid y)$, and therefore makes $\pi_\beta$ stationary, while ergodicity guarantees convergence to this unique stationary distribution from any initial distribution.
